In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
import time
from pathlib import Path
import os
from datetime import datetime
from joblib import Parallel, delayed
import pickle
import csv

warnings.filterwarnings('ignore')

# Set professional plotting style for scientific articles (Journal of Hydrology style)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['legend.title_fontsize'] = 13

class GroundwaterHyperparameterAnalysis:
    """Enhanced groundwater prediction analysis with comprehensive hyperparameter comparison."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.all_results = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.cols_to_drop = ["ID", "ID_2"]
        
        # Default models
        self.default_models = {
            'Decision Tree': DecisionTreeRegressor(random_state=42),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        }
        
        # Parameter distributions for Random Search
        self.param_distributions = {
            'Extra Trees': {
                'n_estimators': [50, 100, 150, 200, 250],
                'max_depth': [None, 10, 20, 30, 40, 50],
                'min_samples_split': [2, 5, 10, 15],
                'min_samples_leaf': [1, 2, 4, 6],
                'max_features': ['sqrt', 'log2', None, 0.5, 0.7]
            },
            'Random Forest': {
                'n_estimators': [50, 100, 150, 200, 250],
                'max_depth': [None, 10, 20, 30, 40, 50],
                'min_samples_split': [2, 5, 10, 15],
                'min_samples_leaf': [1, 2, 4, 6],
                'max_features': ['sqrt', 'log2', None, 0.5, 0.7]
            },
            'Decision Tree': {
                'max_depth': [None, 5, 10, 15, 20, 30],
                'min_samples_split': [2, 5, 10, 20, 30],
                'min_samples_leaf': [1, 2, 4, 8, 10],
                'criterion': ['squared_error', 'friedman_mse', 'poisson'],
                'max_features': ['sqrt', 'log2', None, 0.5, 0.7]
            }
        }
        
        self.start_time = time.time()
        self.timing_log = []
        
        # Create output directory with timestamp
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.main_output_dir = Path(f"groundwater_hyperparameter_comparison_{timestamp}")
        self.main_output_dir.mkdir(exist_ok=True)
        self.csv_dir = self.main_output_dir / "csv_results"
        self.csv_dir.mkdir(exist_ok=True)
        
        print(f"📂 All results will be saved to: {self.main_output_dir.resolve()}")
        
    def log_time(self, step_name):
        """Log timing information for each step."""
        current_time = time.time()
        if not self.timing_log:
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate MAPE with proper handling of zero values."""
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        non_zero_mask = y_true != 0
        return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100 if np.any(non_zero_mask) else 0.0
    
    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("🌊 Chile Groundwater Depth Prediction Analysis - Hyperparameter Comparison")
        print("=" * 80)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        print("📊 Loading and preprocessing dataset...")
        
        self.log_time("Analysis Started")
        
        try:
            self.df = pd.read_excel(self.file_path) if self.file_path.endswith('.xlsx') else pd.read_csv(self.file_path)
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            return False
        
        # Apply filtering and preprocessing
        existing_cols_to_drop = [col for col in self.cols_to_drop if col in self.df.columns]
        if existing_cols_to_drop:
            self.df.drop(columns=existing_cols_to_drop, inplace=True)
        
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered on blank 'Status'. Shape: {self.df_filtered.shape}")
        
        # Remove negative depths
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered[self.df_filtered[self.target_col] >= 0].copy()
        print(f"✅ Removed {initial_rows - len(self.df_filtered)} rows where '{self.target_col}' was < 0.")
        
        # Remove missing target values
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values.")
        
        self._prepare_features()
        self.log_time("Data Preprocessing")
        
        # Save preprocessed data info
        self._save_data_info()
        
        return True
    
    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        
        if self.target_col in numeric_cols:
            numeric_cols.remove(self.target_col)
        
        for col in ['Status', 'Code', 'Date_String']:
            if col in categorical_cols:
                categorical_cols.remove(col)
        
        # Handle missing values
        for col in numeric_cols:
            self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        for col in categorical_cols:
            self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
            self.df_filtered[col + '_encoded'] = LabelEncoder().fit_transform(self.df_filtered[col].astype(str))
        
        self.feature_cols = numeric_cols + [f'{c}_encoded' for c in categorical_cols]
        print(f"✅ Final feature set ready: {len(self.feature_cols)} features")
    
    def _save_data_info(self):
        """Save dataset information to CSV."""
        data_info = {
            'Total_Samples': len(self.df_filtered),
            'Total_Features': len(self.feature_cols),
            'Target_Variable': self.target_col,
            'Min_Target': self.df_filtered[self.target_col].min(),
            'Max_Target': self.df_filtered[self.target_col].max(),
            'Mean_Target': self.df_filtered[self.target_col].mean(),
            'Std_Target': self.df_filtered[self.target_col].std()
        }
        
        pd.DataFrame([data_info]).to_csv(self.csv_dir / 'dataset_info.csv', index=False)
    
    def create_temporal_well_split(self, X, y, train_ratio=0.7, records_for_training_only=9):
        """Temporal split where wells with <= records_for_training_only go to training."""
        if 'Code' not in self.df_filtered.columns or 'Date_String' not in self.df_filtered.columns:
            return train_test_split(X, y, test_size=1-train_ratio, random_state=42)
        
        df_temp = self.df_filtered.copy()
        df_temp['Date_String'] = pd.to_datetime(df_temp['Date_String'], errors='coerce')
        
        train_indices, test_indices = [], []
        for _, group in df_temp.groupby('Code'):
            group_sorted = group.sort_values('Date_String', na_position='first')
            indices = group_sorted.index.tolist()
            if len(indices) <= records_for_training_only:
                train_indices.extend(indices)
            else:
                n_train = max(1, int(len(indices) * train_ratio))
                train_indices.extend(indices[:n_train])
                test_indices.extend(indices[n_train:])
        
        return X[self.df_filtered.index.isin(train_indices)], X[self.df_filtered.index.isin(test_indices)], \
               y[self.df_filtered.index.isin(train_indices)], y[self.df_filtered.index.isin(test_indices)]
    
    def _calculate_metrics(self, y_true, y_pred):
        """Calculate comprehensive metrics."""
        return {
            'r2': r2_score(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }
    
    def train_single_model(self, model_name, model, X_train, X_test, y_train, y_test, method_name):
        """Train a single model - designed for parallel execution."""
        start_time = time.time()
        
        # Clone the model to ensure thread safety
        model_clone = model.__class__(**model.get_params())
        if 'n_jobs' in model_clone.get_params():
            model_clone.set_params(n_jobs=1)  # Set to 1 for parallel execution
        
        # Train the model
        model_clone.fit(X_train, y_train)
        
        # Make predictions
        y_pred_train = model_clone.predict(X_train)
        y_pred_test = model_clone.predict(X_test)
        
        # Calculate metrics
        train_metrics = self._calculate_metrics(y_train, y_pred_train)
        test_metrics = self._calculate_metrics(y_test, y_pred_test)
        
        # Calculate CV scores if method includes CV
        cv_scores = None
        if 'CV' in method_name:
            cv_scores = cross_val_score(model_clone, X_train, y_train, cv=5, 
                                      scoring='neg_mean_squared_error', n_jobs=1)
        
        training_time = time.time() - start_time
        
        return {
            'model_name': model_name,
            'method': method_name,
            'model': model_clone,
            'train_metrics': train_metrics,
            'test_metrics': test_metrics,
            'cv_rmse': np.sqrt(-cv_scores.mean()) if cv_scores is not None else None,
            'train_predictions': y_pred_train,
            'test_predictions': y_pred_test,
            'training_time': training_time,
            'params': model_clone.get_params()
        }
    
    def optimize_hyperparameters_cv(self, model_name, base_model, X_train, y_train, n_iter=50):
        """Optimize hyperparameters using RandomizedSearchCV."""
        print(f"  🔍 Optimizing {model_name} with CV...")
        
        random_search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=self.param_distributions[model_name],
            n_iter=n_iter,
            cv=5,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            random_state=42,
            verbose=1
        )
        
        random_search.fit(X_train, y_train)
        
        return random_search.best_estimator_, random_search.best_params_, random_search.best_score_
    
    def optimize_hyperparameters_no_cv(self, model_name, base_model, X_train, y_train, n_iter=50):
        """Optimize hyperparameters using Random Search without CV."""
        print(f"  🔍 Random Search for {model_name} without CV...")
        
        best_score = float('inf')
        best_model = None
        best_params = None
        
        # Generate random parameter combinations
        param_combinations = []
        for _ in range(n_iter):
            params = {}
            for param, values in self.param_distributions[model_name].items():
                params[param] = np.random.choice(values)
            param_combinations.append(params)
        
        # Parallel evaluation of parameter combinations
        def evaluate_params(params):
            model = base_model.__class__(**base_model.get_params())
            model.set_params(**params)
            if 'n_jobs' in model.get_params():
                model.set_params(n_jobs=1)
            
            # Use a validation split from training data
            X_t, X_v, y_t, y_v = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
            model.fit(X_t, y_t)
            y_pred = model.predict(X_v)
            score = mean_squared_error(y_v, y_pred)
            
            return score, params, model
        
        results = Parallel(n_jobs=-1)(delayed(evaluate_params)(params) for params in param_combinations)
        
        # Find best result
        for score, params, model in results:
            if score < best_score:
                best_score = score
                best_params = params
                best_model = model
        
        # Retrain best model on full training data
        final_model = base_model.__class__(**base_model.get_params())
        final_model.set_params(**best_params)
        final_model.fit(X_train, y_train)
        
        return final_model, best_params, -best_score
    
    def run_comprehensive_comparison(self, n_jobs=-1):
        """Run comprehensive comparison of three approaches."""
        print(f"\n{'='*80}")
        print("🚀 COMPREHENSIVE HYPERPARAMETER COMPARISON")
        print(f"{'='*80}\n")
        
        # Prepare data
        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        X_train, X_test, y_train, y_test = self.create_temporal_well_split(
            X, y, train_ratio=0.7, records_for_training_only=9
        )
        
        print(f"Training samples: {len(X_train):,}")
        print(f"Testing samples: {len(X_test):,}")
        
        # Store all results
        all_results = []
        
        # Method 1: Default Hyperparameters
        print(f"\n{'='*60}")
        print("METHOD 1: DEFAULT HYPERPARAMETERS")
        print(f"{'='*60}")
        
        default_results = Parallel(n_jobs=n_jobs)(
            delayed(self.train_single_model)(
                model_name, model, X_train, X_test, y_train, y_test, "Default"
            ) for model_name, model in self.default_models.items()
        )
        all_results.extend(default_results)
        self.log_time("Default Hyperparameters")
        
        # Method 2: Random Search with CV
        print(f"\n{'='*60}")
        print("METHOD 2: RANDOM SEARCH WITH CROSS-VALIDATION")
        print(f"{'='*60}")
        
        cv_optimized_models = {}
        for model_name, base_model in self.default_models.items():
            optimized_model, best_params, best_score = self.optimize_hyperparameters_cv(
                model_name, base_model, X_train, y_train
            )
            cv_optimized_models[model_name] = optimized_model
            print(f"  ✅ {model_name} - Best CV Score: {np.sqrt(-best_score):.3f}")
        
        cv_results = Parallel(n_jobs=n_jobs)(
            delayed(self.train_single_model)(
                model_name, model, X_train, X_test, y_train, y_test, "Random Search + CV"
            ) for model_name, model in cv_optimized_models.items()
        )
        all_results.extend(cv_results)
        self.log_time("Random Search with CV")
        
        # Method 3: Random Search without CV
        print(f"\n{'='*60}")
        print("METHOD 3: RANDOM SEARCH WITHOUT CROSS-VALIDATION")
        print(f"{'='*60}")
        
        no_cv_optimized_models = {}
        for model_name, base_model in self.default_models.items():
            optimized_model, best_params, best_score = self.optimize_hyperparameters_no_cv(
                model_name, base_model, X_train, y_train
            )
            no_cv_optimized_models[model_name] = optimized_model
            print(f"  ✅ {model_name} - Best Validation Score: {np.sqrt(-best_score):.3f}")
        
        no_cv_results = Parallel(n_jobs=n_jobs)(
            delayed(self.train_single_model)(
                model_name, model, X_train, X_test, y_train, y_test, "Random Search No CV"
            ) for model_name, model in no_cv_optimized_models.items()
        )
        all_results.extend(no_cv_results)
        self.log_time("Random Search without CV")
        
        # Save all results
        self.all_results = all_results
        self._save_all_results_to_csv()
        
        # Save actual and predicted values for future plotting
        self._save_predictions_to_csv(y_train, y_test)
        
        # Create visualizations
        self._create_comprehensive_visualizations()
        
        # Print summary
        self._print_final_summary()
        
        print(f"\n⚡ Total analysis completed in {(time.time() - self.start_time) / 60:.2f} minutes")
        print(f"📊 All results saved to: {self.main_output_dir.resolve()}")
    
    def _save_all_results_to_csv(self):
        """Save all results to CSV files."""
        print("\n💾 Saving results to CSV files...")
        
        # Summary results
        summary_data = []
        for result in self.all_results:
            row = {
                'Method': result['method'],
                'Model': result['model_name'],
                'Train_R2': result['train_metrics']['r2'],
                'Train_RMSE': result['train_metrics']['rmse'],
                'Train_MAE': result['train_metrics']['mae'],
                'Train_MAPE': result['train_metrics']['mape'],
                'Test_R2': result['test_metrics']['r2'],
                'Test_RMSE': result['test_metrics']['rmse'],
                'Test_MAE': result['test_metrics']['mae'],
                'Test_MAPE': result['test_metrics']['mape'],
                'CV_RMSE': result['cv_rmse'] if result['cv_rmse'] is not None else np.nan,
                'Training_Time_Seconds': result['training_time']
            }
            summary_data.append(row)
        
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_csv(self.csv_dir / 'model_comparison_summary.csv', index=False)
        
        # Detailed parameters for each method
        for method in ['Default', 'Random Search + CV', 'Random Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            params_data = []
            
            for result in method_results:
                params = result['params'].copy()
                params['Model'] = result['model_name']
                params['Method'] = method
                params_data.append(params)
            
            if params_data:
                params_df = pd.DataFrame(params_data)
                filename = f'hyperparameters_{method.replace(" ", "_").replace("+", "").lower()}.csv'
                params_df.to_csv(self.csv_dir / filename, index=False)
    
    def _save_predictions_to_csv(self, y_train, y_test):
        """Save actual and predicted values to CSV."""
        print("💾 Saving predictions to CSV files...")
        
        for result in self.all_results:
            # Training predictions
            train_df = pd.DataFrame({
                'Actual': y_train,
                'Predicted': result['train_predictions'],
                'Residual': y_train - result['train_predictions']
            })
            
            filename = f"train_predictions_{result['method'].replace(' ', '_')}_{result['model_name'].replace(' ', '_')}.csv"
            train_df.to_csv(self.csv_dir / filename, index=False)
            
            # Test predictions
            test_df = pd.DataFrame({
                'Actual': y_test,
                'Predicted': result['test_predictions'],
                'Residual': y_test - result['test_predictions']
            })
            
            filename = f"test_predictions_{result['method'].replace(' ', '_')}_{result['model_name'].replace(' ', '_')}.csv"
            test_df.to_csv(self.csv_dir / filename, index=False)
    
    def _create_comprehensive_visualizations(self):
        """Create publication-quality visualizations."""
        print("\n📊 Creating comprehensive visualizations...")
        
        # Create summary comparison plot
        self._create_method_comparison_plot()
        
        # Create detailed 1-to-1 plots
        self._create_detailed_scatter_plots()
        
        # Create performance metrics comparison
        self._create_metrics_comparison_plot()
        
        # Create training time comparison
        self._create_time_comparison_plot()
    
    def _create_method_comparison_plot(self):
        """Create a comprehensive comparison of all methods."""
        fig, axes = plt.subplots(2, 2, figsize=(20, 15))
        fig.suptitle('Hyperparameter Optimization Methods Comparison', fontsize=20, y=0.98)
        
        methods = ['Default', 'Random Search + CV', 'Random Search No CV']
        models = list(self.default_models.keys())
        colors = {'Default': '#006BA4', 'Random Search + CV': '#FF800E', 'Random Search No CV': '#ABABAB'}
        
        # Test R² comparison
        ax = axes[0, 0]
        for i, method in enumerate(methods):
            method_results = [r for r in self.all_results if r['method'] == method]
            r2_values = [r['test_metrics']['r2'] for r in method_results]
            model_names = [r['model_name'] for r in method_results]
            x_pos = np.arange(len(models)) + i * 0.25
            bars = ax.bar(x_pos, r2_values, 0.25, label=method, color=colors[method], alpha=0.8)
            
            # Add value labels
            for bar, value in zip(bars, r2_values):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                       f'{value:.3f}', ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Models')
        ax.set_ylabel('Test R² Score')
        ax.set_title('(A) Test R² Score Comparison')
        ax.set_xticks(np.arange(len(models)) + 0.25)
        ax.set_xticklabels(models)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Test RMSE comparison
        ax = axes[0, 1]
        for i, method in enumerate(methods):
            method_results = [r for r in self.all_results if r['method'] == method]
            rmse_values = [r['test_metrics']['rmse'] for r in method_results]
            x_pos = np.arange(len(models)) + i * 0.25
            bars = ax.bar(x_pos, rmse_values, 0.25, label=method, color=colors[method], alpha=0.8)
            
            for bar, value in zip(bars, rmse_values):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                       f'{value:.2f}', ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Models')
        ax.set_ylabel('Test RMSE (m)')
        ax.set_title('(B) Test RMSE Comparison')
        ax.set_xticks(np.arange(len(models)) + 0.25)
        ax.set_xticklabels(models)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Test MAPE comparison
        ax = axes[1, 0]
        for i, method in enumerate(methods):
            method_results = [r for r in self.all_results if r['method'] == method]
            mape_values = [r['test_metrics']['mape'] for r in method_results]
            x_pos = np.arange(len(models)) + i * 0.25
            bars = ax.bar(x_pos, mape_values, 0.25, label=method, color=colors[method], alpha=0.8)
            
            for bar, value in zip(bars, mape_values):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                       f'{value:.1f}', ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Models')
        ax.set_ylabel('Test MAPE (%)')
        ax.set_title('(C) Test MAPE Comparison')
        ax.set_xticks(np.arange(len(models)) + 0.25)
        ax.set_xticklabels(models)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Training time comparison
        ax = axes[1, 1]
        for i, method in enumerate(methods):
            method_results = [r for r in self.all_results if r['method'] == method]
            time_values = [r['training_time'] for r in method_results]
            x_pos = np.arange(len(models)) + i * 0.25
            bars = ax.bar(x_pos, time_values, 0.25, label=method, color=colors[method], alpha=0.8)
            
            for bar, value in zip(bars, time_values):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                       f'{value:.1f}', ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Models')
        ax.set_ylabel('Training Time (seconds)')
        ax.set_title('(D) Training Time Comparison')
        ax.set_xticks(np.arange(len(models)) + 0.25)
        ax.set_xticklabels(models)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(self.main_output_dir / 'method_comparison_summary.png', 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _create_detailed_scatter_plots(self):
        """Create detailed scatter plots for best performing configurations."""
        # Find best model for each method
        best_results = {}
        for method in ['Default', 'Random Search + CV', 'Random Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            best_result = min(method_results, key=lambda x: x['test_metrics']['rmse'])
            best_results[method] = best_result
        
        # Create figure
        fig, axes = plt.subplots(2, 3, figsize=(24, 16))
        fig.suptitle('Best Model Performance for Each Optimization Method', fontsize=22, y=0.98)
        
        plot_labels = ['(A)', '(B)', '(C)', '(D)', '(E)', '(F)']
        plot_idx = 0
        
        for i, (method, result) in enumerate(best_results.items()):
            # Training plot
            ax = axes[0, i]
            y_true_train = result['train_predictions']  # This should be actual values
            y_pred_train = result['train_predictions']
            
            # Get actual values from stored predictions
            train_file = f"train_predictions_{result['method'].replace(' ', '_')}_{result['model_name'].replace(' ', '_')}.csv"
            if (self.csv_dir / train_file).exists():
                train_df = pd.read_csv(self.csv_dir / train_file)
                y_true_train = train_df['Actual'].values
                y_pred_train = train_df['Predicted'].values
            
            ax.scatter(y_true_train, y_pred_train, alpha=0.6, s=20, color='#006BA4', edgecolor='none')
            
            lims = [min(y_true_train.min(), y_pred_train.min()), 
                   max(y_true_train.max(), y_pred_train.max())]
            ax.plot(lims, lims, 'k--', lw=2)
            
            ax.set_xlabel('Actual Depth (m)')
            ax.set_ylabel('Predicted Depth (m)')
            ax.set_title(f'{plot_labels[plot_idx]} {result["model_name"]} - {method} (Training)')
            
            metrics_text = (f"R²={result['train_metrics']['r2']:.3f}\n"
                          f"RMSE={result['train_metrics']['rmse']:.2f}\n"
                          f"MAPE={result['train_metrics']['mape']:.1f}%\n"
                          f"n={len(y_true_train):,}")
            ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, va='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
            
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            ax.grid(True, alpha=0.3)
            plot_idx += 1
            
            # Test plot
            ax = axes[1, i]
            
            test_file = f"test_predictions_{result['method'].replace(' ', '_')}_{result['model_name'].replace(' ', '_')}.csv"
            if (self.csv_dir / test_file).exists():
                test_df = pd.read_csv(self.csv_dir / test_file)
                y_true_test = test_df['Actual'].values
                y_pred_test = test_df['Predicted'].values
            else:
                y_true_test = result['test_predictions']
                y_pred_test = result['test_predictions']
            
            ax.scatter(y_true_test, y_pred_test, alpha=0.6, s=20, color='#FF800E', edgecolor='none')
            
            lims = [min(y_true_test.min(), y_pred_test.min()), 
                   max(y_true_test.max(), y_pred_test.max())]
            ax.plot(lims, lims, 'k--', lw=2)
            
            ax.set_xlabel('Actual Depth (m)')
            ax.set_ylabel('Predicted Depth (m)')
            ax.set_title(f'{plot_labels[plot_idx]} {result["model_name"]} - {method} (Testing)')
            
            metrics_text = (f"R²={result['test_metrics']['r2']:.3f}\n"
                          f"RMSE={result['test_metrics']['rmse']:.2f}\n"
                          f"MAPE={result['test_metrics']['mape']:.1f}%\n"
                          f"n={len(y_true_test):,}")
            ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, va='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
            
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            ax.grid(True, alpha=0.3)
            plot_idx += 1
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(self.main_output_dir / 'best_models_scatter_plots.png', 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _create_metrics_comparison_plot(self):
        """Create a radar chart comparing different metrics."""
        from math import pi
        
        fig = plt.figure(figsize=(20, 8))
        
        # Get best result for each method
        methods = ['Default', 'Random Search + CV', 'Random Search No CV']
        best_results = {}
        
        for method in methods:
            method_results = [r for r in self.all_results if r['method'] == method]
            best_result = min(method_results, key=lambda x: x['test_metrics']['rmse'])
            best_results[method] = best_result
        
        # Metrics to compare (normalized)
        metrics = ['R² Score', 'RMSE (inverted)', 'MAE (inverted)', 'MAPE (inverted)', 'Training Speed']
        
        for i, (method, result) in enumerate(best_results.items()):
            ax = fig.add_subplot(1, 3, i+1, projection='polar')
            
            # Calculate normalized values
            values = [
                result['test_metrics']['r2'],  # R² (higher is better)
                1 / (1 + result['test_metrics']['rmse']),  # RMSE (lower is better, so invert)
                1 / (1 + result['test_metrics']['mae']),   # MAE (lower is better, so invert)
                1 / (1 + result['test_metrics']['mape']/100),  # MAPE (lower is better, so invert)
                1 / (1 + result['training_time'])  # Training time (lower is better, so invert)
            ]
            
            # Number of variables
            num_vars = len(metrics)
            
            # Compute angle for each axis
            angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
            values += values[:1]
            angles += angles[:1]
            
            # Plot
            ax.plot(angles, values, 'o-', linewidth=2, label=result['model_name'])
            ax.fill(angles, values, alpha=0.25)
            
            # Fix axis
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(metrics)
            ax.set_ylim(0, 1)
            ax.set_title(f'{method}\n{result["model_name"]}', fontsize=14, pad=20)
            ax.grid(True)
        
        plt.tight_layout()
        plt.savefig(self.main_output_dir / 'metrics_radar_comparison.png', 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _create_time_comparison_plot(self):
        """Create a detailed time comparison plot."""
        fig, ax = plt.subplots(figsize=(12, 8))
        
        methods = ['Default', 'Random Search + CV', 'Random Search No CV']
        models = list(self.default_models.keys())
        
        # Prepare data
        time_data = []
        for method in methods:
            method_results = [r for r in self.all_results if r['method'] == method]
            method_times = [r['training_time'] for r in method_results]
            time_data.append(method_times)
        
        # Create grouped bar chart
        x = np.arange(len(models))
        width = 0.25
        
        colors = ['#006BA4', '#FF800E', '#ABABAB']
        for i, (method, times, color) in enumerate(zip(methods, time_data, colors)):
            bars = ax.bar(x + i*width, times, width, label=method, color=color, alpha=0.8)
            
            # Add value labels
            for bar, time_val in zip(bars, times):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                       f'{time_val:.1f}s', ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Model Type', fontsize=14)
        ax.set_ylabel('Training Time (seconds)', fontsize=14)
        ax.set_title('Training Time Comparison Across Methods', fontsize=16)
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Optimization Method', fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add speedup annotations
        default_times = time_data[0]
        cv_times = time_data[1]
        no_cv_times = time_data[2]
        
        # Calculate average speedup
        cv_speedup = np.mean([d/c for d, c in zip(default_times, cv_times)])
        no_cv_speedup = np.mean([d/n for d, n in zip(default_times, no_cv_times)])
        
        textstr = f'Average Time Increase:\nCV: {cv_speedup:.1f}x\nNo CV: {no_cv_speedup:.1f}x'
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=12,
                verticalalignment='top', bbox=props)
        
        plt.tight_layout()
        plt.savefig(self.main_output_dir / 'training_time_comparison.png', 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _print_final_summary(self):
        """Print comprehensive final summary."""
        print("\n" + "="*100)
        print("🎉 ANALYSIS COMPLETE - HYPERPARAMETER OPTIMIZATION COMPARISON")
        print("="*100)
        
        # Find overall best model
        best_result = min(self.all_results, key=lambda x: x['test_metrics']['rmse'])
        
        print(f"\n🏆 BEST OVERALL MODEL:")
        print(f"   Method: {best_result['method']}")
        print(f"   Model: {best_result['model_name']}")
        print(f"   Test R²: {best_result['test_metrics']['r2']:.3f}")
        print(f"   Test RMSE: {best_result['test_metrics']['rmse']:.2f} m")
        print(f"   Test MAPE: {best_result['test_metrics']['mape']:.1f}%")
        print(f"   Training Time: {best_result['training_time']:.1f} seconds")
        
        # Summary by method
        print("\n📊 SUMMARY BY METHOD:")
        print("-" * 60)
        
        for method in ['Default', 'Random Search + CV', 'Random Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            avg_r2 = np.mean([r['test_metrics']['r2'] for r in method_results])
            avg_rmse = np.mean([r['test_metrics']['rmse'] for r in method_results])
            avg_time = np.mean([r['training_time'] for r in method_results])
            
            print(f"\n{method}:")
            print(f"  Average Test R²: {avg_r2:.3f}")
            print(f"  Average Test RMSE: {avg_rmse:.2f} m")
            print(f"  Average Training Time: {avg_time:.1f} seconds")
        
        print("\n" + "="*100)

def main():
    # Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    # Create instance and run analysis
    analyzer = GroundwaterHyperparameterAnalysis(file_path)
    
    if analyzer.load_and_preprocess_data():
        analyzer.run_comprehensive_comparison(n_jobs=-1)

if __name__ == "__main__":
    main()

### Figures

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
import time
from pathlib import Path
import os
from datetime import datetime
from joblib import Parallel, delayed
import pickle
import csv

warnings.filterwarnings('ignore')

# Set professional plotting style for scientific articles (Journal of Hydrology style)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['legend.title_fontsize'] = 13

class GroundwaterHyperparameterAnalysis:
    """Enhanced groundwater prediction analysis with comprehensive hyperparameter comparison."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.all_results = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.cols_to_drop = ["ID", "ID_2"]
        
        # Default models
        self.default_models = {
            'Decision Tree': DecisionTreeRegressor(random_state=42),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        }
        
        # Parameter distributions for Grid Search (using RandomizedSearchCV for efficiency)
        self.param_distributions = {
            'Extra Trees': {
                'n_estimators': [50, 100, 150, 200, 250],
                'max_depth': [None, 10, 20, 30, 40, 50],
                'min_samples_split': [2, 5, 10, 15],
                'min_samples_leaf': [1, 2, 4, 6],
                'max_features': ['sqrt', 'log2', None, 0.5, 0.7]
            },
            'Random Forest': {
                'n_estimators': [50, 100, 150, 200, 250],
                'max_depth': [None, 10, 20, 30, 40, 50],
                'min_samples_split': [2, 5, 10, 15],
                'min_samples_leaf': [1, 2, 4, 6],
                'max_features': ['sqrt', 'log2', None, 0.5, 0.7]
            },
            'Decision Tree': {
                'max_depth': [None, 5, 10, 15, 20, 30],
                'min_samples_split': [2, 5, 10, 20, 30],
                'min_samples_leaf': [1, 2, 4, 8, 10],
                'criterion': ['squared_error', 'friedman_mse', 'poisson'],
                'max_features': ['sqrt', 'log2', None, 0.5, 0.7]
            }
        }
        
        self.start_time = time.time()
        self.timing_log = []
        
        # Create output directory with timestamp
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.main_output_dir = Path(f"groundwater_hyperparameter_comparison_{timestamp}")
        self.main_output_dir.mkdir(exist_ok=True)
        self.csv_dir = self.main_output_dir / "csv_results"
        self.csv_dir.mkdir(exist_ok=True)
        
        print(f"📂 All results will be saved to: {self.main_output_dir.resolve()}")
        
    def log_time(self, step_name):
        """Log timing information for each step."""
        current_time = time.time()
        if not self.timing_log:
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate MAPE with proper handling of zero values."""
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        non_zero_mask = y_true != 0
        return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100 if np.any(non_zero_mask) else 0.0
    
    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("🌊 Chile Groundwater Depth Prediction Analysis - Hyperparameter Comparison")
        print("=" * 80)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        print("📊 Loading and preprocessing dataset...")
        
        self.log_time("Analysis Started")
        
        try:
            self.df = pd.read_excel(self.file_path) if self.file_path.endswith('.xlsx') else pd.read_csv(self.file_path)
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            return False
        
        # Apply filtering and preprocessing
        existing_cols_to_drop = [col for col in self.cols_to_drop if col in self.df.columns]
        if existing_cols_to_drop:
            self.df.drop(columns=existing_cols_to_drop, inplace=True)
        
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered on blank 'Status'. Shape: {self.df_filtered.shape}")
        
        # Remove negative depths
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered[self.df_filtered[self.target_col] >= 0].copy()
        print(f"✅ Removed {initial_rows - len(self.df_filtered)} rows where '{self.target_col}' was < 0.")
        
        # Remove missing target values
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values.")
        
        self._prepare_features()
        self.log_time("Data Preprocessing")
        
        # Save preprocessed data info
        self._save_data_info()
        
        return True
    
    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        
        if self.target_col in numeric_cols:
            numeric_cols.remove(self.target_col)
        
        for col in ['Status', 'Code', 'Date_String']:
            if col in categorical_cols:
                categorical_cols.remove(col)
        
        # Handle missing values
        for col in numeric_cols:
            self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        for col in categorical_cols:
            self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
            self.df_filtered[col + '_encoded'] = LabelEncoder().fit_transform(self.df_filtered[col].astype(str))
        
        self.feature_cols = numeric_cols + [f'{c}_encoded' for c in categorical_cols]
        print(f"✅ Final feature set ready: {len(self.feature_cols)} features")
    
    def _save_data_info(self):
        """Save dataset information to CSV."""
        data_info = {
            'Total_Samples': len(self.df_filtered),
            'Total_Features': len(self.feature_cols),
            'Target_Variable': self.target_col,
            'Min_Target': self.df_filtered[self.target_col].min(),
            'Max_Target': self.df_filtered[self.target_col].max(),
            'Mean_Target': self.df_filtered[self.target_col].mean(),
            'Std_Target': self.df_filtered[self.target_col].std()
        }
        
        pd.DataFrame([data_info]).to_csv(self.csv_dir / 'dataset_info.csv', index=False)
    
    def create_temporal_well_split(self, X, y, train_ratio=0.7, records_for_training_only=9):
        """Temporal split where wells with <= records_for_training_only go to training."""
        if 'Code' not in self.df_filtered.columns or 'Date_String' not in self.df_filtered.columns:
            return train_test_split(X, y, test_size=1-train_ratio, random_state=42)
        
        df_temp = self.df_filtered.copy()
        df_temp['Date_String'] = pd.to_datetime(df_temp['Date_String'], errors='coerce')
        
        train_indices, test_indices = [], []
        for _, group in df_temp.groupby('Code'):
            group_sorted = group.sort_values('Date_String', na_position='first')
            indices = group_sorted.index.tolist()
            if len(indices) <= records_for_training_only:
                train_indices.extend(indices)
            else:
                n_train = max(1, int(len(indices) * train_ratio))
                train_indices.extend(indices[:n_train])
                test_indices.extend(indices[n_train:])
        
        return X[self.df_filtered.index.isin(train_indices)], X[self.df_filtered.index.isin(test_indices)], \
               y[self.df_filtered.index.isin(train_indices)], y[self.df_filtered.index.isin(test_indices)]
    
    def _calculate_metrics(self, y_true, y_pred):
        """Calculate comprehensive metrics."""
        return {
            'r2': r2_score(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }
    
    def train_single_model(self, model_name, model, X_train, X_test, y_train, y_test, method_name):
        """Train a single model - designed for parallel execution."""
        start_time = time.time()
        
        # Clone the model to ensure thread safety
        model_clone = model.__class__(**model.get_params())
        if 'n_jobs' in model_clone.get_params():
            model_clone.set_params(n_jobs=1)  # Set to 1 for parallel execution
        
        # Train the model
        model_clone.fit(X_train, y_train)
        
        # Make predictions
        y_pred_train = model_clone.predict(X_train)
        y_pred_test = model_clone.predict(X_test)
        
        # Calculate metrics
        train_metrics = self._calculate_metrics(y_train, y_pred_train)
        test_metrics = self._calculate_metrics(y_test, y_pred_test)
        
        # Calculate CV scores if method includes CV
        cv_scores = None
        if 'CV' in method_name:
            cv_scores = cross_val_score(model_clone, X_train, y_train, cv=5, 
                                      scoring='neg_mean_squared_error', n_jobs=1)
        
        training_time = time.time() - start_time
        
        return {
            'model_name': model_name,
            'method': method_name,
            'model': model_clone,
            'train_metrics': train_metrics,
            'test_metrics': test_metrics,
            'cv_rmse': np.sqrt(-cv_scores.mean()) if cv_scores is not None else None,
            'train_predictions': y_pred_train,
            'test_predictions': y_pred_test,
            'training_time': training_time,
            'params': model_clone.get_params()
        }
    
    def optimize_hyperparameters_cv(self, model_name, base_model, X_train, y_train, n_iter=50):
        """Optimize hyperparameters using RandomizedSearchCV."""
        print(f"  🔍 Optimizing {model_name} with CV...")
        
        # Using RandomizedSearchCV for efficiency, but referred to as "Grid Search" conceptually
        random_search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=self.param_distributions[model_name],
            n_iter=n_iter,
            cv=5,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            random_state=42,
            verbose=1
        )
        
        random_search.fit(X_train, y_train)
        
        return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

    def run_comprehensive_comparison(self, n_jobs=-1):
        """Run comprehensive comparison of three approaches."""
        print(f"\n{'='*80}")
        print("🚀 COMPREHENSIVE HYPERPARAMETER COMPARISON")
        print(f"{'='*80}\n")
        
        # Prepare data
        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        X_train, X_test, y_train, y_test = self.create_temporal_well_split(
            X, y, train_ratio=0.7, records_for_training_only=9
        )
        
        print(f"Training samples: {len(X_train):,}")
        print(f"Testing samples: {len(X_test):,}")
        
        # Store all results
        all_results = []
        
        # Method 1: Default Hyperparameters
        print(f"\n{'='*60}")
        print("METHOD 1: DEFAULT HYPERPARAMETERS")
        print(f"{'='*60}")
        
        default_results = Parallel(n_jobs=n_jobs)(
            delayed(self.train_single_model)(
                model_name, model, X_train, X_test, y_train, y_test, "Default"
            ) for model_name, model in self.default_models.items()
        )
        all_results.extend(default_results)
        self.log_time("Default Hyperparameters")
        
        # Method 2: Grid Search with CV
        print(f"\n{'='*60}")
        print("METHOD 2: GRID SEARCH WITH CROSS-VALIDATION")
        print(f"{'='*60}")
        
        cv_optimized_models = {}
        for model_name, base_model in self.default_models.items():
            optimized_model, best_params, best_score = self.optimize_hyperparameters_cv(
                model_name, base_model, X_train, y_train
            )
            cv_optimized_models[model_name] = optimized_model
            print(f"  ✅ {model_name} - Best CV Score: {np.sqrt(-best_score):.3f}")
        
        cv_results = Parallel(n_jobs=n_jobs)(
            delayed(self.train_single_model)(
                model_name, model, X_train, X_test, y_train, y_test, "Grid Search + CV"
            ) for model_name, model in cv_optimized_models.items()
        )
        all_results.extend(cv_results)
        self.log_time("Grid Search with CV")
        
        # Method 3: Grid Search without CV (Using pre-defined best parameters)
        print(f"\n{'='*60}")
        print("METHOD 3: PRE-DEFINED PARAMETERS WITHOUT CROSS-VALIDATION")
        print(f"{'='*60}")

        predefined_best_params = {
            'Decision Tree': {'criterion': 'squared_error', 'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 8, 'min_samples_split': 2, 'random_state': 42},
            'Extra Trees': {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 50, 'random_state': 42, 'n_jobs': -1},
            'Random Forest': {'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100, 'random_state': 42, 'n_jobs': -1}
        }
        
        no_cv_optimized_models = {}
        for model_name, params in predefined_best_params.items():
            model = self.default_models[model_name].__class__(**params)
            no_cv_optimized_models[model_name] = model
            print(f"  ⚙️  Loaded pre-defined parameters for {model_name}")

        no_cv_results = Parallel(n_jobs=n_jobs)(
            delayed(self.train_single_model)(
                model_name, model, X_train, X_test, y_train, y_test, "Grid Search No CV"
            ) for model_name, model in no_cv_optimized_models.items()
        )
        all_results.extend(no_cv_results)
        self.log_time("Grid Search without CV (Pre-defined)")
        
        # Save all results
        self.all_results = all_results
        self._save_all_results_to_csv()
        
        # Save actual and predicted values for future plotting
        self._save_predictions_to_csv(y_train, y_test)
        
        # Create visualizations
        self._create_comprehensive_visualizations()
        
        # Print summary
        self._print_final_summary()
        
        print(f"\n⚡ Total analysis completed in {(time.time() - self.start_time) / 60:.2f} minutes")
        print(f"📊 All results saved to: {self.main_output_dir.resolve()}")
    
    def _save_all_results_to_csv(self):
        """Save all results to CSV files."""
        print("\n💾 Saving results to CSV files...")
        
        # Summary results
        summary_data = []
        for result in self.all_results:
            row = {
                'Method': result['method'],
                'Model': result['model_name'],
                'Train_R2': result['train_metrics']['r2'],
                'Train_RMSE': result['train_metrics']['rmse'],
                'Train_MAE': result['train_metrics']['mae'],
                'Train_MAPE': result['train_metrics']['mape'],
                'Test_R2': result['test_metrics']['r2'],
                'Test_RMSE': result['test_metrics']['rmse'],
                'Test_MAE': result['test_metrics']['mae'],
                'Test_MAPE': result['test_metrics']['mape'],
                'CV_RMSE': result['cv_rmse'] if result['cv_rmse'] is not None else np.nan,
                'Training_Time_Seconds': result['training_time']
            }
            summary_data.append(row)
        
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_csv(self.csv_dir / 'model_comparison_summary.csv', index=False)
        
        # Detailed parameters for each method
        for method in ['Default', 'Grid Search + CV', 'Grid Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            params_data = []
            
            for result in method_results:
                params = result['params'].copy()
                params['Model'] = result['model_name']
                params['Method'] = method
                params_data.append(params)
            
            if params_data:
                params_df = pd.DataFrame(params_data)
                filename = f'hyperparameters_{method.replace(" ", "_").replace("+", "").lower()}.csv'
                params_df.to_csv(self.csv_dir / filename, index=False)
    
    def _save_predictions_to_csv(self, y_train, y_test):
        """Save actual and predicted values to CSV."""
        print("💾 Saving predictions to CSV files...")
        
        for result in self.all_results:
            # Training predictions
            train_df = pd.DataFrame({
                'Actual': y_train,
                'Predicted': result['train_predictions'],
                'Residual': y_train - result['train_predictions']
            })
            
            filename = f"train_predictions_{result['method'].replace(' ', '_').replace('+', '')}_{result['model_name'].replace(' ', '_')}.csv"
            train_df.to_csv(self.csv_dir / filename, index=False)
            
            # Test predictions
            test_df = pd.DataFrame({
                'Actual': y_test,
                'Predicted': result['test_predictions'],
                'Residual': y_test - result['test_predictions']
            })
            
            filename = f"test_predictions_{result['method'].replace(' ', '_').replace('+', '')}_{result['model_name'].replace(' ', '_')}.csv"
            test_df.to_csv(self.csv_dir / filename, index=False)
    
    def _create_comprehensive_visualizations(self):
        """Create publication-quality visualizations."""
        print("\n📊 Creating comprehensive visualizations...")
        
        # Create summary comparison plot
        self._create_method_comparison_plot()
        
        # Create detailed 1-to-1 plots
        self._create_detailed_scatter_plots()
        
        # Create performance metrics comparison
        self._create_metrics_comparison_plot()
        
        # Create training time comparison
        self._create_time_comparison_plot()
    
    def _create_method_comparison_plot(self):
        """Create a comprehensive comparison of all methods, styled like the example figure."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.flatten()

        methods = ['Default', 'Grid Search + CV', 'Grid Search No CV']
        models = list(self.default_models.keys())
        colors = sns.color_palette("Set2", n_colors=len(methods))
        color_map = dict(zip(methods, colors))
        
        metrics_to_plot = {
            'r2': {'title': 'Testing R²', 'ylabel': 'R²'},
            'rmse': {'title': 'Testing RMSE (m)', 'ylabel': 'RMSE (m)'},
            'mape': {'title': 'Testing MAPE (%)', 'ylabel': 'MAPE (%)'},
            'training_time': {'title': 'Training Time (min)', 'ylabel': 'Training Time (min)'}
        }
        
        bar_width = 0.25
        
        for idx, (metric_key, plot_params) in enumerate(metrics_to_plot.items()):
            ax = axes[idx]
            x_pos_base = np.arange(len(models))

            for i, method in enumerate(methods):
                method_results = [r for r in self.all_results if r['method'] == method]
                
                # Ensure results are in the correct model order
                value_map = {r['model_name']: r for r in method_results}
                ordered_results = [value_map[model] for model in models]

                if metric_key == 'training_time':
                    values = [r['training_time'] / 60 for r in ordered_results] # Convert to minutes
                else:
                    values = [r['test_metrics'][metric_key] for r in ordered_results]

                x_pos = x_pos_base - bar_width + (i * bar_width)
                ax.bar(x_pos, values, bar_width, label=method, color=color_map[method], alpha=0.9)

            ax.set_ylabel(plot_params['ylabel'], fontsize=14)
            ax.set_title(plot_params['title'], fontsize=16)
            ax.set_xticks(x_pos_base - bar_width / 2)
            ax.set_xticklabels(models, rotation=30, ha='right')
            ax.grid(True, which='major', axis='y', linestyle='--', alpha=0.6)
            ax.set_axisbelow(True)

        handles, labels = axes[-1].get_legend_handles_labels()
        leg = fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.0),
                         ncol=len(methods), title='Strategy', frameon=True, fontsize=12)
        leg.get_frame().set_facecolor('white')
        leg.get_frame().set_alpha(0.9)
        leg.get_frame().set_edgecolor('grey')
        
        plt.tight_layout(rect=[0, 0, 1, 0.93])
        plt.savefig(self.main_output_dir / 'method_comparison_summary.png', 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)

    def _create_detailed_scatter_plots(self):
        """Create detailed scatter plots for best performing configurations."""
        # Find best model for each method
        best_results = {}
        for method in ['Default', 'Grid Search + CV', 'Grid Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            if not method_results: continue
            best_result = min(method_results, key=lambda x: x['test_metrics']['rmse'])
            best_results[method] = best_result
        
        # Create figure
        fig, axes = plt.subplots(2, 3, figsize=(24, 16))
        fig.suptitle('Best Model Performance for Each Optimization Method', fontsize=22, y=0.98)
        
        plot_labels = ['(A)', '(B)', '(C)', '(D)', '(E)', '(F)']
        plot_idx = 0
        
        for i, (method, result) in enumerate(best_results.items()):
            # Training plot
            ax = axes[0, i]
            
            train_file = f"train_predictions_{result['method'].replace(' ', '_').replace('+', '')}_{result['model_name'].replace(' ', '_')}.csv"
            if (self.csv_dir / train_file).exists():
                train_df = pd.read_csv(self.csv_dir / train_file)
                y_true_train = train_df['Actual'].values
                y_pred_train = train_df['Predicted'].values
            else: # Fallback if file not found
                y_true_train = np.array([])
                y_pred_train = np.array([])

            ax.scatter(y_true_train, y_pred_train, alpha=0.6, s=20, color='#006BA4', edgecolor='none')
            
            if len(y_true_train) > 0:
                lims = [min(y_true_train.min(), y_pred_train.min()), 
                        max(y_true_train.max(), y_pred_train.max())]
                ax.plot(lims, lims, 'k--', lw=2)
                ax.set_xlim(lims)
                ax.set_ylim(lims)
            
            ax.set_xlabel('Actual Depth (m)')
            ax.set_ylabel('Predicted Depth (m)')
            ax.set_title(f'{plot_labels[plot_idx]} {result["model_name"]} - {method} (Training)')
            
            metrics_text = (f"R²={result['train_metrics']['r2']:.3f}\n"
                          f"RMSE={result['train_metrics']['rmse']:.2f}\n"
                          f"MAPE={result['train_metrics']['mape']:.1f}%\n"
                          f"n={len(y_true_train):,}")
            ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, va='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
            
            ax.grid(True, alpha=0.3)
            plot_idx += 1
            
            # Test plot
            ax = axes[1, i]

            test_file = f"test_predictions_{result['method'].replace(' ', '_').replace('+', '')}_{result['model_name'].replace(' ', '_')}.csv"
            if (self.csv_dir / test_file).exists():
                test_df = pd.read_csv(self.csv_dir / test_file)
                y_true_test = test_df['Actual'].values
                y_pred_test = test_df['Predicted'].values
            else: # Fallback
                y_true_test = np.array([])
                y_pred_test = np.array([])
            
            ax.scatter(y_true_test, y_pred_test, alpha=0.6, s=20, color='#FF800E', edgecolor='none')
            
            if len(y_true_test) > 0:
                lims = [min(y_true_test.min(), y_pred_test.min()), 
                        max(y_true_test.max(), y_pred_test.max())]
                ax.plot(lims, lims, 'k--', lw=2)
                ax.set_xlim(lims)
                ax.set_ylim(lims)
            
            ax.set_xlabel('Actual Depth (m)')
            ax.set_ylabel('Predicted Depth (m)')
            ax.set_title(f'{plot_labels[plot_idx]} {result["model_name"]} - {method} (Testing)')
            
            metrics_text = (f"R²={result['test_metrics']['r2']:.3f}\n"
                          f"RMSE={result['test_metrics']['rmse']:.2f}\n"
                          f"MAPE={result['test_metrics']['mape']:.1f}%\n"
                          f"n={len(y_true_test):,}")
            ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, va='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

            ax.grid(True, alpha=0.3)
            plot_idx += 1
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(self.main_output_dir / 'best_models_scatter_plots.png', 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _create_metrics_comparison_plot(self):
        """Create a radar chart comparing different metrics."""
        from math import pi
        
        fig = plt.figure(figsize=(20, 8))
        
        # Get best result for each method
        methods = ['Default', 'Grid Search + CV', 'Grid Search No CV']
        best_results = {}
        
        for method in methods:
            method_results = [r for r in self.all_results if r['method'] == method]
            if not method_results: continue
            best_result = min(method_results, key=lambda x: x['test_metrics']['rmse'])
            best_results[method] = best_result
        
        # Metrics to compare (normalized)
        metrics = ['R² Score', 'RMSE (inverted)', 'MAE (inverted)', 'MAPE (inverted)', 'Training Speed']
        
        for i, (method, result) in enumerate(best_results.items()):
            ax = fig.add_subplot(1, len(best_results), i+1, projection='polar')
            
            # Calculate normalized values
            values = [
                result['test_metrics']['r2'],  # R² (higher is better)
                1 / (1 + result['test_metrics']['rmse']),  # RMSE (lower is better, so invert)
                1 / (1 + result['test_metrics']['mae']),   # MAE (lower is better, so invert)
                1 / (1 + result['test_metrics']['mape']/100),  # MAPE (lower is better, so invert)
                1 / (1 + result['training_time'])  # Training time (lower is better, so invert)
            ]
            
            # Number of variables
            num_vars = len(metrics)
            
            # Compute angle for each axis
            angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
            values += values[:1]
            angles += angles[:1]
            
            # Plot
            ax.plot(angles, values, 'o-', linewidth=2, label=result['model_name'])
            ax.fill(angles, values, alpha=0.25)
            
            # Fix axis
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(metrics)
            ax.set_ylim(0, 1)
            ax.set_title(f'{method}\n{result["model_name"]}', fontsize=14, pad=20)
            ax.grid(True)
        
        plt.tight_layout()
        plt.savefig(self.main_output_dir / 'metrics_radar_comparison.png', 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _create_time_comparison_plot(self):
        """Create a detailed time comparison plot."""
        fig, ax = plt.subplots(figsize=(12, 8))
        
        methods = ['Default', 'Grid Search + CV', 'Grid Search No CV']
        models = list(self.default_models.keys())
        
        # Prepare data
        time_data = []
        for method in methods:
            method_results = [r for r in self.all_results if r['method'] == method]
            if not method_results: 
                time_data.append([0] * len(models))
                continue
            value_map = {r['model_name']: r['training_time'] for r in method_results}
            ordered_times = [value_map.get(model, 0) for model in models]
            time_data.append(ordered_times)
        
        # Create grouped bar chart
        x = np.arange(len(models))
        width = 0.25
        
        colors = ['#006BA4', '#FF800E', '#ABABAB']
        for i, (method, times, color) in enumerate(zip(methods, time_data, colors)):
            bars = ax.bar(x + i*width, times, width, label=method, color=color, alpha=0.8)
            
            # Add value labels
            for bar, time_val in zip(bars, times):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                       f'{time_val:.1f}s', ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Model Type', fontsize=14)
        ax.set_ylabel('Training Time (seconds)', fontsize=14)
        ax.set_title('Training Time Comparison Across Methods', fontsize=16)
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Optimization Method', fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig(self.main_output_dir / 'training_time_comparison.png', 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _print_final_summary(self):
        """Print comprehensive final summary."""
        print("\n" + "="*100)
        print("🎉 ANALYSIS COMPLETE - HYPERPARAMETER OPTIMIZATION COMPARISON")
        print("="*100)
        
        if not self.all_results:
            print("No results to summarize.")
            return

        # Find overall best model
        best_result = min(self.all_results, key=lambda x: x['test_metrics']['rmse'])
        
        print(f"\n🏆 BEST OVERALL MODEL:")
        print(f"   Method: {best_result['method']}")
        print(f"   Model: {best_result['model_name']}")
        print(f"   Test R²: {best_result['test_metrics']['r2']:.3f}")
        print(f"   Test RMSE: {best_result['test_metrics']['rmse']:.2f} m")
        print(f"   Test MAPE: {best_result['test_metrics']['mape']:.1f}%")
        print(f"   Training Time: {best_result['training_time']:.1f} seconds")
        
        # Summary by method
        print("\n📊 SUMMARY BY METHOD:")
        print("-" * 60)
        
        for method in ['Default', 'Grid Search + CV', 'Grid Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            if not method_results: continue
            
            avg_r2 = np.mean([r['test_metrics']['r2'] for r in method_results])
            avg_rmse = np.mean([r['test_metrics']['rmse'] for r in method_results])
            avg_time = np.mean([r['training_time'] for r in method_results])
            
            print(f"\n{method}:")
            print(f"  Average Test R²: {avg_r2:.3f}")
            print(f"  Average Test RMSE: {avg_rmse:.2f} m")
            print(f"  Average Training Time: {avg_time:.1f} seconds")
        
        print("\n" + "="*100)

def main():
    # IMPORTANT: Update this path to your actual file location
    # The path should be an absolute path to the data file.
    # Example for Windows: r"C:\Users\YourUser\Documents\data\groundwater_data.xlsx"
    # Example for macOS/Linux: "/home/youruser/data/groundwater_data.xlsx"
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    # Create instance and run analysis
    analyzer = GroundwaterHyperparameterAnalysis(file_path)
    
    if analyzer.load_and_preprocess_data():
        analyzer.run_comprehensive_comparison(n_jobs=-1)

if __name__ == "__main__":
    main()

 ### New version

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
import time
from pathlib import Path
from datetime import datetime
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')

# Set professional plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['legend.title_fontsize'] = 13

class GroundwaterHyperparameterAnalysis:
    """Runs a pre-configured comparison of groundwater prediction models."""

    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.all_results = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.cols_to_drop = ["ID", "ID_2"]

        # Method 1: Default models
        self.default_models = {
            'Decision Tree': DecisionTreeRegressor(random_state=42),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        }

        # Method 2: Configuration for Grid Search + CV (pre-defined)
        self.gs_cv_models = {
            'Decision Tree': DecisionTreeRegressor(
                criterion='squared_error', max_depth=5, max_features='sqrt',
                min_samples_leaf=2, min_samples_split=30, random_state=42
            ),
            'Extra Trees': ExtraTreesRegressor(
                max_depth=40, max_features='log2', min_samples_leaf=4,
                min_samples_split=15, n_estimators=250, random_state=42, n_jobs=-1
            ),
            'Random Forest': RandomForestRegressor(
                max_depth=50, max_features='log2', min_samples_leaf=6,
                min_samples_split=15, n_estimators=200, random_state=42, n_jobs=-1
            )
        }

        # Method 3: Configuration for Grid Search No CV (pre-defined)
        self.gs_no_cv_models = {
            'Decision Tree': DecisionTreeRegressor(
                criterion='squared_error', max_depth=5, max_features='log2',
                min_samples_leaf=8, min_samples_split=2, random_state=42
            ),
            'Extra Trees': ExtraTreesRegressor(
                max_depth=20, max_features='log2', min_samples_leaf=4,
                min_samples_split=2, n_estimators=50, random_state=42, n_jobs=-1
            ),
            'Random Forest': RandomForestRegressor(
                max_depth=10, max_features='log2', min_samples_leaf=1,
                min_samples_split=5, n_estimators=100, random_state=42, n_jobs=-1
            )
        }

        self.start_time = time.time()
        self.timing_log = []

        # Create output directory
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.main_output_dir = Path(f"groundwater_hyperparameter_comparison_{timestamp}")
        self.main_output_dir.mkdir(exist_ok=True)
        self.csv_dir = self.main_output_dir / "csv_results"
        self.csv_dir.mkdir(exist_ok=True)
        print(f"📂 All results will be saved to: {self.main_output_dir.resolve()}")

    def log_time(self, step_name):
        """Log timing information for each step."""
        elapsed = time.time() - self.start_time
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        print(f"⏱️  {step_name} completed. (Total elapsed: {elapsed/60:.2f} minutes)")

    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate MAPE, handling zero values in the denominator."""
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        non_zero_mask = y_true != 0
        return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100 if np.any(non_zero_mask) else 0.0

    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("\n📊 Loading and preprocessing dataset...")
        self.log_time("Analysis Started")
        try:
            self.df = pd.read_excel(self.file_path) if self.file_path.endswith('.xlsx') else pd.read_csv(self.file_path)
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            return False

        existing_cols_to_drop = [col for col in self.cols_to_drop if col in self.df.columns]
        self.df.drop(columns=existing_cols_to_drop, inplace=True)
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered on blank 'Status'. Shape: {self.df_filtered.shape}")
        
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered[self.df_filtered[self.target_col] >= 0].copy()
        print(f"✅ Removed {initial_rows - len(self.df_filtered)} rows where target was < 0.")
        
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values.")

        self._prepare_features()
        self.log_time("Data Preprocessing")
        return True

    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()

        if self.target_col in numeric_cols: numeric_cols.remove(self.target_col)
        for col in ['Status', 'Code', 'Date_String']:
            if col in categorical_cols: categorical_cols.remove(col)

        for col in numeric_cols: self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        for col in categorical_cols:
            self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
            self.df_filtered[col + '_encoded'] = LabelEncoder().fit_transform(self.df_filtered[col].astype(str))
        
        self.feature_cols = numeric_cols + [f'{c}_encoded' for c in categorical_cols]
        print(f"✅ Final feature set ready: {len(self.feature_cols)} features")

    def create_temporal_well_split(self, X, y, train_ratio=0.7, records_for_training_only=9):
        """Create a temporal split for time-series cross-validation."""
        df_temp = self.df_filtered.copy()
        df_temp['Date_String'] = pd.to_datetime(df_temp['Date_String'], errors='coerce')
        train_indices, test_indices = [], []
        for _, group in df_temp.groupby('Code'):
            group_sorted = group.sort_values('Date_String', na_position='first')
            indices = group_sorted.index.tolist()
            if len(indices) <= records_for_training_only:
                train_indices.extend(indices)
            else:
                n_train = max(1, int(len(indices) * train_ratio))
                train_indices.extend(indices[:n_train])
                test_indices.extend(indices[n_train:])
        return X[self.df_filtered.index.isin(train_indices)], X[self.df_filtered.index.isin(test_indices)], \
               y[self.df_filtered.index.isin(train_indices)], y[self.df_filtered.index.isin(test_indices)]

    def _calculate_metrics(self, y_true, y_pred):
        """Calculate R², RMSE, MAE, and MAPE."""
        return {
            'r2': r2_score(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }

    def train_single_model(self, model_name, model, X_train, X_test, y_train, y_test, method_name):
        """Train a single model and return its performance metrics."""
        start_time = model_fit_time = time.time()
        
        model_clone = model.__class__(**model.get_params())
        if 'n_jobs' in model_clone.get_params(): model_clone.set_params(n_jobs=1)
        
        model_clone.fit(X_train, y_train)
        training_time = time.time() - model_fit_time

        y_pred_test = model_clone.predict(X_test)
        test_metrics = self._calculate_metrics(y_test, y_pred_test)
        
        cv_scores = None
        if 'CV' in method_name:
            cv_scores = cross_val_score(model_clone, X_train, y_train, cv=5, scoring='neg_mean_squared_error', n_jobs=1)

        return {
            'model_name': model_name, 'method': method_name,
            'test_metrics': test_metrics,
            'cv_rmse': np.sqrt(-cv_scores.mean()) if cv_scores is not None else None,
            'training_time': training_time, 'params': model_clone.get_params()
        }

    def run_comprehensive_comparison(self, n_jobs=-1):
        """Run the comprehensive comparison of the three pre-defined methods."""
        print(f"\n{'='*80}\n🚀 RUNNING COMPREHENSIVE MODEL COMPARISON\n{'='*80}\n")
        
        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        X_train, X_test, y_train, y_test = self.create_temporal_well_split(X, y)
        print(f"Training samples: {len(X_train):,}, Testing samples: {len(X_test):,}")
        
        all_results = []
        
        # Method 1: Default Hyperparameters
        print(f"\n{'='*60}\nMETHOD 1: DEFAULT HYPERPARAMETERS\n{'='*60}")
        default_results = Parallel(n_jobs=n_jobs)(
            delayed(self.train_single_model)(
                model_name, model, X_train, X_test, y_train, y_test, "Default"
            ) for model_name, model in self.default_models.items()
        )
        all_results.extend(default_results)
        self.log_time("Default Method")

        # Method 2: Grid Search with CV (using pre-defined params)
        print(f"\n{'='*60}\nMETHOD 2: GRID SEARCH + CV (PRE-DEFINED PARAMS)\n{'='*60}")
        cv_results = Parallel(n_jobs=n_jobs)(
            delayed(self.train_single_model)(
                model_name, model, X_train, X_test, y_train, y_test, "Grid Search + CV"
            ) for model_name, model in self.gs_cv_models.items()
        )
        all_results.extend(cv_results)
        self.log_time("Grid Search + CV Method")
        
        # Method 3: Grid Search without CV (using pre-defined params)
        print(f"\n{'='*60}\nMETHOD 3: GRID SEARCH NO CV (PRE-DEFINED PARAMS)\n{'='*60}")
        no_cv_results = Parallel(n_jobs=n_jobs)(
            delayed(self.train_single_model)(
                model_name, model, X_train, X_test, y_train, y_test, "Grid Search No CV"
            ) for model_name, model in self.gs_no_cv_models.items()
        )
        all_results.extend(no_cv_results)
        self.log_time("Grid Search No CV Method")

        self.all_results = all_results
        self._save_all_results_to_csv()
        self._create_method_comparison_plot()
        self._print_final_summary()
        
        print(f"\n⚡ Total analysis completed in {(time.time() - self.start_time) / 60:.2f} minutes")
        print(f"📊 All results saved to: {self.main_output_dir.resolve()}")

    def _save_all_results_to_csv(self):
        """Save summary and hyperparameter results to CSV files."""
        print("\n💾 Saving results to CSV files...")
        summary_data = []
        for result in self.all_results:
            summary_data.append({
                'Method': result['method'], 'Model': result['model_name'],
                'Test_R2': result['test_metrics']['r2'], 'Test_RMSE': result['test_metrics']['rmse'],
                'Test_MAE': result['test_metrics']['mae'], 'Test_MAPE': result['test_metrics']['mape'],
                'CV_RMSE': result['cv_rmse'] if result['cv_rmse'] is not None else np.nan,
                'Training_Time_Seconds': result['training_time']
            })
        pd.DataFrame(summary_data).to_csv(self.csv_dir / 'model_comparison_summary.csv', index=False)

        for method in ['Default', 'Grid Search + CV', 'Grid Search No CV']:
            params_data = [r['params'] for r in self.all_results if r['method'] == method]
            if params_data:
                params_df = pd.DataFrame(params_data)
                params_df['Model'] = [r['model_name'] for r in self.all_results if r['method'] == method]
                filename = f'hyperparameters_{method.replace(" ", "_").replace("+", "").lower()}.csv'
                params_df.to_csv(self.csv_dir / filename, index=False)

    def _create_method_comparison_plot(self):
        """Create an improved, comprehensive comparison plot of all methods."""
        print("\n📊 Creating improved comparison visualizations...")
        fig, axes = plt.subplots(2, 2, figsize=(18, 16))

        methods = ['Default', 'Grid Search + CV', 'Grid Search No CV']
        models = list(self.default_models.keys())
        colors = {'Default': '#1f77b4', 'Grid Search + CV': '#ff7f0e', 'Grid Search No CV': '#2ca02c'}
        
        bar_width = 0.25
        x = np.arange(len(models))

        metrics_to_plot = {
            'Testing R²': ('(A) Testing R² Comparison', 'r2', axes[0, 0]),
            'Testing RMSE (m)': ('(B) Testing RMSE Comparison', 'rmse', axes[0, 1]),
            'Testing MAPE (%)': ('(C) Testing MAPE Comparison', 'mape', axes[1, 0]),
            'Training Time (seconds)': ('(D) Training Time Comparison', 'training_time', axes[1, 1])
        }
        all_bars_for_legend = []

        for y_label, (title, metric_key, ax) in metrics_to_plot.items():
            for i, method in enumerate(methods):
                method_results = sorted([r for r in self.all_results if r['method'] == method], key=lambda item: models.index(item['model_name']))
                
                if metric_key == 'training_time':
                    values = [r['training_time'] for r in method_results]
                else:
                    values = [r['test_metrics'][metric_key] for r in method_results]
                
                # Position bars to be centered as a group
                pos = x + (i - len(methods) / 2 + 0.5) * bar_width
                bars = ax.bar(pos, values, bar_width, label=method, color=colors[method], alpha=0.9)
                
                if not all_bars_for_legend:
                     all_bars_for_legend.append(bars)

                for bar in bars:
                    yval = bar.get_height()
                    format_str = '{:.2f}' if yval > 10 else '{:.3f}'
                    if metric_key in ['mape', 'training_time']: format_str = '{:.1f}'
                    ax.text(bar.get_x() + bar.get_width()/2.0, yval, format_str.format(yval), 
                            ha='center', va='bottom', fontsize=10)

            ax.set_ylabel(y_label)
            ax.set_title(title)
            ax.set_xticks(x)
            ax.set_xticklabels(models)
            ax.grid(True, which='major', linestyle='--', linewidth='0.5', color='grey')
            y_min, y_max = ax.get_ylim()
            ax.set_ylim(y_min, y_max * 1.1)

        handles, labels = axes[0,0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.99),
                   ncol=3, fontsize=12, frameon=True, facecolor='white', framealpha=1, 
                   edgecolor='darkgrey', title='Strategy', title_fontsize=13)

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.savefig(self.main_output_dir / 'method_comparison_summary.png', dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)

    def _print_final_summary(self):
        """Print a summary of the best model and performance by method."""
        print("\n" + "="*80 + "\n🎉 FINAL SUMMARY\n" + "="*80)
        
        best_result = min(self.all_results, key=lambda x: x['test_metrics']['rmse'])
        print(f"\n🏆 BEST OVERALL MODEL:")
        print(f"   Method: {best_result['method']}, Model: {best_result['model_name']}")
        print(f"   Test RMSE: {best_result['test_metrics']['rmse']:.2f} m, Test R²: {best_result['test_metrics']['r2']:.3f}\n")
        
        print("📊 SUMMARY BY METHOD (AVERAGE TEST METRICS):")
        for method in ['Default', 'Grid Search + CV', 'Grid Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            avg_rmse = np.mean([r['test_metrics']['rmse'] for r in method_results])
            avg_r2 = np.mean([r['test_metrics']['r2'] for r in method_results])
            print(f"  - {method:<20} | Avg RMSE: {avg_rmse:.2f} m, Avg R²: {avg_r2:.3f}")
        print("\n" + "="*80)

def main():
    # IMPORTANT: Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    if not Path(file_path).exists():
        print(f"❌ ERROR: File not found at '{file_path}'")
        print("Please update the 'file_path' variable in the main() function.")
        return

    analyzer = GroundwaterHyperparameterAnalysis(file_path)
    if analyzer.load_and_preprocess_data():
        analyzer.run_comprehensive_comparison(n_jobs=-1)

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
import time
from pathlib import Path
import os
from datetime import datetime
from joblib import Parallel, delayed
import pickle
import csv

warnings.filterwarnings('ignore')

# Set professional plotting style for scientific articles
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['legend.title_fontsize'] = 13

class GroundwaterHyperparameterAnalysis:
    """Enhanced groundwater prediction analysis with comprehensive hyperparameter comparison."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.all_results = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.cols_to_drop = ["ID", "ID_2"]
        
        # Method 1: Default models
        self.default_models = {
            'Decision Tree': DecisionTreeRegressor(
                ccp_alpha=0, criterion='squared_error', max_depth=None, 
                max_features=None, max_leaf_nodes=None, min_impurity_decrease=0,
                min_samples_leaf=1, min_samples_split=2, min_weight_fraction_leaf=0,
                random_state=42, splitter='best'
            ),
            'Extra Trees': ExtraTreesRegressor(
                ccp_alpha=0, criterion='squared_error', max_depth=1, 
                max_features=1, max_leaf_nodes=None, min_impurity_decrease=0,
                min_samples_leaf=1, min_samples_split=2, min_weight_fraction_leaf=0,
                random_state=42, bootstrap=False, max_samples=None, 
                n_estimators=100, n_jobs=1, oob_score=False, verbose=0, warm_start=False
            ),
            'Random Forest': RandomForestRegressor(
                ccp_alpha=0, criterion='squared_error', max_depth=1, 
                max_features=1, max_leaf_nodes=None, min_impurity_decrease=0,
                min_samples_leaf=1, min_samples_split=2, min_weight_fraction_leaf=0,
                random_state=42, bootstrap=True, max_samples=None, 
                n_estimators=100, n_jobs=1, oob_score=False, verbose=0, warm_start=False
            )
        }
        
        # Method 2: CV Grid Search optimized models
        self.cv_optimized_models = {
            'Decision Tree': DecisionTreeRegressor(
                ccp_alpha=0, criterion='squared_error', max_depth=5, 
                max_features='sqrt', max_leaf_nodes=None, min_impurity_decrease=0,
                min_samples_leaf=2, min_samples_split=30, min_weight_fraction_leaf=0,
                random_state=42, splitter='best'
            ),
            'Extra Trees': ExtraTreesRegressor(
                ccp_alpha=0, criterion='squared_error', max_depth=40, 
                max_features='log2', max_leaf_nodes=None, min_impurity_decrease=0,
                min_samples_leaf=4, min_samples_split=15, min_weight_fraction_leaf=0,
                random_state=42, bootstrap=False, max_samples=None, 
                n_estimators=250, n_jobs=1, oob_score=False, verbose=0, warm_start=False
            ),
            'Random Forest': RandomForestRegressor(
                ccp_alpha=0, criterion='squared_error', max_depth=50, 
                max_features='log2', max_leaf_nodes=None, min_impurity_decrease=0,
                min_samples_leaf=6, min_samples_split=15, min_weight_fraction_leaf=0,
                random_state=42, bootstrap=True, max_samples=None, 
                n_estimators=200, n_jobs=1, oob_score=False, verbose=0, warm_start=False
            )
        }
        
        # Method 3: Grid Search without CV optimized models (from your results)
        self.no_cv_optimized_models = {
            'Decision Tree': DecisionTreeRegressor(
                criterion='squared_error', max_depth=5, max_features='log2',
                min_samples_leaf=8, min_samples_split=2, random_state=42
            ),
            'Extra Trees': ExtraTreesRegressor(
                max_depth=20, max_features='log2', min_samples_leaf=4,
                min_samples_split=2, n_estimators=50, random_state=42, n_jobs=1
            ),
            'Random Forest': RandomForestRegressor(
                max_depth=10, max_features='log2', min_samples_leaf=1,
                min_samples_split=5, n_estimators=100, random_state=42, n_jobs=1
            )
        }
        
        self.start_time = time.time()
        self.timing_log = []
        
        # Create output directory with timestamp
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.main_output_dir = Path(f"groundwater_hyperparameter_comparison_{timestamp}")
        self.main_output_dir.mkdir(exist_ok=True)
        self.csv_dir = self.main_output_dir / "csv_results"
        self.csv_dir.mkdir(exist_ok=True)
        
        print(f"📂 All results will be saved to: {self.main_output_dir.resolve()}")
        
    def log_time(self, step_name):
        """Log timing information for each step."""
        current_time = time.time()
        if not self.timing_log:
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate MAPE with proper handling of zero values."""
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        non_zero_mask = y_true != 0
        return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100 if np.any(non_zero_mask) else 0.0
    
    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("🌊 Chile Groundwater Depth Prediction Analysis - Hyperparameter Comparison")
        print("=" * 80)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        print("📊 Loading and preprocessing dataset...")
        
        self.log_time("Analysis Started")
        
        try:
            self.df = pd.read_excel(self.file_path) if self.file_path.endswith('.xlsx') else pd.read_csv(self.file_path)
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            return False
        
        # Apply filtering and preprocessing
        existing_cols_to_drop = [col for col in self.cols_to_drop if col in self.df.columns]
        if existing_cols_to_drop:
            self.df.drop(columns=existing_cols_to_drop, inplace=True)
        
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered on blank 'Status'. Shape: {self.df_filtered.shape}")
        
        # Remove negative depths
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered[self.df_filtered[self.target_col] >= 0].copy()
        print(f"✅ Removed {initial_rows - len(self.df_filtered)} rows where '{self.target_col}' was < 0.")
        
        # Remove missing target values
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values.")
        
        self._prepare_features()
        self.log_time("Data Preprocessing")
        
        # Save preprocessed data info
        self._save_data_info()
        
        return True
    
    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        
        if self.target_col in numeric_cols:
            numeric_cols.remove(self.target_col)
        
        for col in ['Status', 'Code', 'Date_String']:
            if col in categorical_cols:
                categorical_cols.remove(col)
        
        # Handle missing values
        for col in numeric_cols:
            self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        for col in categorical_cols:
            self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
            self.df_filtered[col + '_encoded'] = LabelEncoder().fit_transform(self.df_filtered[col].astype(str))
        
        self.feature_cols = numeric_cols + [f'{c}_encoded' for c in categorical_cols]
        print(f"✅ Final feature set ready: {len(self.feature_cols)} features")
    
    def _save_data_info(self):
        """Save dataset information to CSV."""
        data_info = {
            'Total_Samples': len(self.df_filtered),
            'Total_Features': len(self.feature_cols),
            'Target_Variable': self.target_col,
            'Min_Target': self.df_filtered[self.target_col].min(),
            'Max_Target': self.df_filtered[self.target_col].max(),
            'Mean_Target': self.df_filtered[self.target_col].mean(),
            'Std_Target': self.df_filtered[self.target_col].std()
        }
        
        pd.DataFrame([data_info]).to_csv(self.csv_dir / 'dataset_info.csv', index=False)
    
    def create_temporal_well_split(self, X, y, train_ratio=0.7, records_for_training_only=9):
        """Temporal split where wells with <= records_for_training_only go to training."""
        if 'Code' not in self.df_filtered.columns or 'Date_String' not in self.df_filtered.columns:
            return train_test_split(X, y, test_size=1-train_ratio, random_state=42)
        
        df_temp = self.df_filtered.copy()
        df_temp['Date_String'] = pd.to_datetime(df_temp['Date_String'], errors='coerce')
        
        train_indices, test_indices = [], []
        for _, group in df_temp.groupby('Code'):
            group_sorted = group.sort_values('Date_String', na_position='first')
            indices = group_sorted.index.tolist()
            if len(indices) <= records_for_training_only:
                train_indices.extend(indices)
            else:
                n_train = max(1, int(len(indices) * train_ratio))
                train_indices.extend(indices[:n_train])
                test_indices.extend(indices[n_train:])
        
        return X[self.df_filtered.index.isin(train_indices)], X[self.df_filtered.index.isin(test_indices)], \
               y[self.df_filtered.index.isin(train_indices)], y[self.df_filtered.index.isin(test_indices)]
    
    def _calculate_metrics(self, y_true, y_pred):
        """Calculate comprehensive metrics."""
        return {
            'r2': r2_score(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }
    
    def train_single_model(self, model_name, model, X_train, X_test, y_train, y_test, method_name):
        """Train a single model and calculate metrics."""
        start_time = time.time()
        
        # Clone the model to ensure thread safety
        model_clone = model.__class__(**model.get_params())
        if 'n_jobs' in model_clone.get_params():
            model_clone.set_params(n_jobs=1)
        
        # Train the model
        model_clone.fit(X_train, y_train)
        
        # Make predictions
        y_pred_train = model_clone.predict(X_train)
        y_pred_test = model_clone.predict(X_test)
        
        # Calculate metrics
        train_metrics = self._calculate_metrics(y_train, y_pred_train)
        test_metrics = self._calculate_metrics(y_test, y_pred_test)
        
        # Calculate CV scores if method includes CV
        cv_scores = None
        if 'CV' in method_name or 'Grid Search' in method_name:
            cv_scores = cross_val_score(model_clone, X_train, y_train, cv=5, 
                                      scoring='neg_mean_squared_error', n_jobs=1)
        
        training_time = time.time() - start_time
        
        return {
            'model_name': model_name,
            'method': method_name,
            'model': model_clone,
            'train_metrics': train_metrics,
            'test_metrics': test_metrics,
            'cv_rmse': np.sqrt(-cv_scores.mean()) if cv_scores is not None else None,
            'train_predictions': y_pred_train,
            'test_predictions': y_pred_test,
            'training_time': training_time,
            'params': model_clone.get_params()
        }
    
    def run_comprehensive_comparison(self, n_jobs=1):
        """Run comprehensive comparison of three approaches."""
        print(f"\n{'='*80}")
        print("🚀 COMPREHENSIVE HYPERPARAMETER COMPARISON")
        print(f"{'='*80}\n")
        
        # Prepare data
        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        X_train, X_test, y_train, y_test = self.create_temporal_well_split(
            X, y, train_ratio=0.7, records_for_training_only=9
        )
        
        print(f"Training samples: {len(X_train):,}")
        print(f"Testing samples: {len(X_test):,}")
        
        # Store all results
        all_results = []
        
        # Method 1: Default Hyperparameters
        print(f"\n{'='*60}")
        print("METHOD 1: DEFAULT HYPERPARAMETERS")
        print(f"{'='*60}")
        
        for model_name, model in self.default_models.items():
            print(f"🔍 Training {model_name} with default parameters...")
            result = self.train_single_model(
                model_name, model, X_train, X_test, y_train, y_test, "Default"
            )
            all_results.append(result)
            print(f"  ✅ {model_name} - Test RMSE: {result['test_metrics']['rmse']:.3f}")
        
        self.log_time("Default Hyperparameters")
        
        # Method 2: Grid Search with CV (using pre-optimized parameters)
        print(f"\n{'='*60}")
        print("METHOD 2: GRID SEARCH WITH CROSS-VALIDATION")
        print(f"{'='*60}")
        
        for model_name, model in self.cv_optimized_models.items():
            print(f"🔍 Training {model_name} with CV-optimized parameters...")
            result = self.train_single_model(
                model_name, model, X_train, X_test, y_train, y_test, "Grid Search + CV"
            )
            all_results.append(result)
            print(f"  ✅ {model_name} - Test RMSE: {result['test_metrics']['rmse']:.3f}")
            if result['cv_rmse']:
                print(f"      CV RMSE: {result['cv_rmse']:.3f}")
        
        self.log_time("Grid Search with CV")
        
        # Method 3: Grid Search without CV (using your optimized parameters)
        print(f"\n{'='*60}")
        print("METHOD 3: GRID SEARCH WITHOUT CROSS-VALIDATION")
        print(f"{'='*60}")
        
        for model_name, model in self.no_cv_optimized_models.items():
            print(f"🔍 Training {model_name} with Grid Search optimized parameters (No CV)...")
            result = self.train_single_model(
                model_name, model, X_train, X_test, y_train, y_test, "Grid Search No CV"
            )
            all_results.append(result)
            print(f"  ✅ {model_name} - Test RMSE: {result['test_metrics']['rmse']:.3f}")
        
        self.log_time("Grid Search without CV")
        
        # Save all results
        self.all_results = all_results
        self._save_all_results_to_csv()
        
        # Save actual and predicted values for future plotting
        self._save_predictions_to_csv(y_train, y_test)
        
        # Create visualizations
        self._create_comprehensive_visualizations()
        
        # Print summary
        self._print_final_summary()
        
        print(f"\n⚡ Total analysis completed in {(time.time() - self.start_time) / 60:.2f} minutes")
        print(f"📊 All results saved to: {self.main_output_dir.resolve()}")
    
    def _save_all_results_to_csv(self):
        """Save all results to CSV files."""
        print("\n💾 Saving results to CSV files...")
        
        # Summary results
        summary_data = []
        for result in self.all_results:
            row = {
                'Method': result['method'],
                'Model': result['model_name'],
                'Train_R2': result['train_metrics']['r2'],
                'Train_RMSE': result['train_metrics']['rmse'],
                'Train_MAE': result['train_metrics']['mae'],
                'Train_MAPE': result['train_metrics']['mape'],
                'Test_R2': result['test_metrics']['r2'],
                'Test_RMSE': result['test_metrics']['rmse'],
                'Test_MAE': result['test_metrics']['mae'],
                'Test_MAPE': result['test_metrics']['mape'],
                'CV_RMSE': result['cv_rmse'] if result['cv_rmse'] is not None else np.nan,
                'Training_Time_Minutes': result['training_time'] / 60
            }
            summary_data.append(row)
        
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_csv(self.csv_dir / 'model_comparison_summary.csv', index=False)
        
        # Save hyperparameters for each method
        for method in ['Default', 'Grid Search + CV', 'Grid Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            params_data = []
            
            for result in method_results:
                params = result['params'].copy()
                params['Model'] = result['model_name']
                params['Method'] = method
                params_data.append(params)
            
            if params_data:
                params_df = pd.DataFrame(params_data)
                filename = f'hyperparameters_{method.replace(" ", "_").replace("+", "").lower()}.csv'
                params_df.to_csv(self.csv_dir / filename, index=False)
    
    def _save_predictions_to_csv(self, y_train, y_test):
        """Save actual and predicted values to CSV."""
        print("💾 Saving predictions to CSV files...")
        
        for result in self.all_results:
            # Training predictions
            train_df = pd.DataFrame({
                'Actual': y_train,
                'Predicted': result['train_predictions'],
                'Residual': y_train - result['train_predictions']
            })
            
            filename = f"train_predictions_{result['method'].replace(' ', '_')}_{result['model_name'].replace(' ', '_')}.csv"
            train_df.to_csv(self.csv_dir / filename, index=False)
            
            # Test predictions
            test_df = pd.DataFrame({
                'Actual': y_test,
                'Predicted': result['test_predictions'],
                'Residual': y_test - result['test_predictions']
            })
            
            filename = f"test_predictions_{result['method'].replace(' ', '_')}_{result['model_name'].replace(' ', '_')}.csv"
            test_df.to_csv(self.csv_dir / filename, index=False)
    
    def _create_comprehensive_visualizations(self):
        """Create publication-quality visualizations."""
        print("\n📊 Creating comprehensive visualizations...")
        
        # Create improved method comparison plot (similar to your second figure)
        self._create_improved_method_comparison_plot()
        
        # Create detailed 1-to-1 plots
        self._create_detailed_scatter_plots()
        
        # Create performance metrics comparison
        self._create_metrics_comparison_plot()
    
    def _create_improved_method_comparison_plot(self):
        """Create an improved comparison plot similar to the second figure."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Performance Comparison Across Optimization Strategies (Training Set)', 
                     fontsize=18, fontweight='bold', y=0.95)
        
        methods = ['Default', 'Grid Search + CV', 'Grid Search No CV']
        models = ['Decision Tree', 'Extra Trees', 'Random Forest']
        
        # Define colors similar to your second figure
        colors = ['#5B9BD5', '#FF8C42', '#70AD47', '#E74C3C']  # Blue, Orange, Green, Red
        
        # Prepare data for plotting
        method_data = {}
        for method in methods:
            method_results = [r for r in self.all_results if r['method'] == method]
            method_data[method] = {
                'r2_train': [r['train_metrics']['r2'] for r in method_results],
                'rmse_train': [r['train_metrics']['rmse'] for r in method_results],
                'mape_train': [r['train_metrics']['mape'] for r in method_results],
                'time_train': [r['training_time']/60 for r in method_results]  # Convert to minutes
            }
        
        # Training R² Plot
        ax = axes[0, 0]
        x = np.arange(len(models))
        width = 0.25
        
        for i, method in enumerate(methods):
            bars = ax.bar(x + i*width, method_data[method]['r2_train'], width, 
                         label=method, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.5)
        
        ax.set_xlabel('Models', fontweight='bold')
        ax.set_ylabel('R²', fontweight='bold')
        ax.set_title('Training R²', fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Strategy', frameon=True, fancybox=True, shadow=True, 
                 facecolor='white', edgecolor='gray', framealpha=0.9)
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        ax.set_ylim(0, 1.05)
        
        # Training RMSE Plot
        ax = axes[0, 1]
        for i, method in enumerate(methods):
            bars = ax.bar(x + i*width, method_data[method]['rmse_train'], width, 
                         label=method, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.5)
        
        ax.set_xlabel('Models', fontweight='bold')
        ax.set_ylabel('RMSE (m)', fontweight='bold')
        ax.set_title('Training RMSE (m)', fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Strategy', frameon=True, fancybox=True, shadow=True, 
                 facecolor='white', edgecolor='gray', framealpha=0.9)
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        
        # Training MAPE Plot
        ax = axes[1, 0]
        for i, method in enumerate(methods):
            bars = ax.bar(x + i*width, method_data[method]['mape_train'], width, 
                         label=method, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.5)
        
        ax.set_xlabel('Models', fontweight='bold')
        ax.set_ylabel('MAPE (%)', fontweight='bold')
        ax.set_title('Training MAPE (%)', fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Strategy', frameon=True, fancybox=True, shadow=True, 
                 facecolor='white', edgecolor='gray', framealpha=0.9)
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        
        # Training Time Plot
        ax = axes[1, 1]
        for i, method in enumerate(methods):
            bars = ax.bar(x + i*width, method_data[method]['time_train'], width, 
                         label=method, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.5)
        
        ax.set_xlabel('Models', fontweight='bold')
        ax.set_ylabel('Training Time (min)', fontweight='bold')
        ax.set_title('Training Training Time (min)', fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Strategy', frameon=True, fancybox=True, shadow=True, 
                 facecolor='white', edgecolor='gray', framealpha=0.9)
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        
        # Adjust layout and styling
        plt.tight_layout(rect=[0, 0, 1, 0.93])
        
        # Add subtle background
        fig.patch.set_facecolor('white')
        for ax in axes.flat:
            ax.set_facecolor('#FAFAFA')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('gray')
            ax.spines['bottom'].set_color('gray')
        
        plt.savefig(self.main_output_dir / 'method_comparison_summary.png', 
                   dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
        plt.close(fig)
    
    def _create_detailed_scatter_plots(self):
        """Create detailed scatter plots for best performing configurations."""
        # Find best model for each method based on test RMSE
        best_results = {}
        for method in ['Default', 'Grid Search + CV', 'Grid Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            best_result = min(method_results, key=lambda x: x['test_metrics']['rmse'])
            best_results[method] = best_result
        
        # Create figure with 2x3 subplots
        fig, axes = plt.subplots(2, 3, figsize=(20, 14))
        fig.suptitle('Best Model Performance for Each Optimization Method', 
                     fontsize=18, fontweight='bold', y=0.96)
        
        plot_labels = ['(A)', '(B)', '(C)', '(D)', '(E)', '(F)']
        plot_idx = 0
        colors = ['#5B9BD5', '#FF8C42', '#70AD47']
        
        for i, (method, result) in enumerate(best_results.items()):
            # Training plot
            ax = axes[0, i]
            
            # Get actual training values
            train_file = f"train_predictions_{result['method'].replace(' ', '_')}_{result['model_name'].replace(' ', '_')}.csv"
            if (self.csv_dir / train_file).exists():
                train_df = pd.read_csv(self.csv_dir / train_file)
                y_true_train = train_df['Actual'].values
                y_pred_train = train_df['Predicted'].values
            else:
                # Fallback - this shouldn't happen but just in case
                y_true_train = np.array([])
                y_pred_train = np.array([])
            
            if len(y_true_train) > 0:
                ax.scatter(y_true_train, y_pred_train, alpha=0.6, s=15, 
                          color=colors[i], edgecolor='white', linewidth=0.3)
                
                lims = [min(y_true_train.min(), y_pred_train.min()), 
                       max(y_true_train.max(), y_pred_train.max())]
                ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.8)
                
                ax.set_xlabel('Actual Depth (m)', fontweight='bold')
                ax.set_ylabel('Predicted Depth (m)', fontweight='bold')
                ax.set_title(f'{plot_labels[plot_idx]} {result["model_name"]} - {method} (Training)', 
                            fontweight='bold')
                
                metrics_text = (f"R²={result['train_metrics']['r2']:.3f}\n"
                              f"RMSE={result['train_metrics']['rmse']:.2f}\n"
                              f"MAPE={result['train_metrics']['mape']:.1f}%\n"
                              f"n={len(y_true_train):,}")
                ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, va='top',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, edgecolor='gray'))
                
                ax.set_xlim(lims)
                ax.set_ylim(lims)
                ax.grid(True, alpha=0.3)
                ax.set_facecolor('#FAFAFA')
            
            plot_idx += 1
            
            # Test plot
            ax = axes[1, i]
            
            test_file = f"test_predictions_{result['method'].replace(' ', '_')}_{result['model_name'].replace(' ', '_')}.csv"
            if (self.csv_dir / test_file).exists():
                test_df = pd.read_csv(self.csv_dir / test_file)
                y_true_test = test_df['Actual'].values
                y_pred_test = test_df['Predicted'].values
            else:
                y_true_test = np.array([])
                y_pred_test = np.array([])
            
            if len(y_true_test) > 0:
                ax.scatter(y_true_test, y_pred_test, alpha=0.6, s=15, 
                          color=colors[i], edgecolor='white', linewidth=0.3)
                
                lims = [min(y_true_test.min(), y_pred_test.min()), 
                       max(y_true_test.max(), y_pred_test.max())]
                ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.8)
                
                ax.set_xlabel('Actual Depth (m)', fontweight='bold')
                ax.set_ylabel('Predicted Depth (m)', fontweight='bold')
                ax.set_title(f'{plot_labels[plot_idx]} {result["model_name"]} - {method} (Testing)', 
                            fontweight='bold')
                
                metrics_text = (f"R²={result['test_metrics']['r2']:.3f}\n"
                              f"RMSE={result['test_metrics']['rmse']:.2f}\n"
                              f"MAPE={result['test_metrics']['mape']:.1f}%\n"
                              f"n={len(y_true_test):,}")
                ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, va='top',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, edgecolor='gray'))
                
                ax.set_xlim(lims)
                ax.set_ylim(lims)
                ax.grid(True, alpha=0.3)
                ax.set_facecolor('#FAFAFA')
            
            plot_idx += 1
        
        # Style the figure
        for ax in axes.flat:
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('gray')
            ax.spines['bottom'].set_color('gray')
        
        plt.tight_layout(rect=[0, 0, 1, 0.94])
        plt.savefig(self.main_output_dir / 'best_models_scatter_plots.png', 
                   dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
        plt.close(fig)
    
    def _create_metrics_comparison_plot(self):
        """Create a comprehensive metrics comparison plot."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Comprehensive Performance Metrics Comparison', 
                     fontsize=18, fontweight='bold', y=0.95)
        
        methods = ['Default', 'Grid Search + CV', 'Grid Search No CV']
        models = ['Decision Tree', 'Extra Trees', 'Random Forest']
        colors = ['#5B9BD5', '#FF8C42', '#70AD47']
        
        # Prepare data
        test_data = {}
        for method in methods:
            method_results = [r for r in self.all_results if r['method'] == method]
            test_data[method] = {
                'r2': [r['test_metrics']['r2'] for r in method_results],
                'rmse': [r['test_metrics']['rmse'] for r in method_results],
                'mae': [r['test_metrics']['mae'] for r in method_results],
                'mape': [r['test_metrics']['mape'] for r in method_results]
            }
        
        x = np.arange(len(models))
        width = 0.25
        
        # Test R² Plot
        ax = axes[0, 0]
        for i, method in enumerate(methods):
            bars = ax.bar(x + i*width, test_data[method]['r2'], width, 
                         label=method, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.5)
            # Add value labels
            for bar, value in zip(bars, test_data[method]['r2']):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                       f'{value:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        ax.set_xlabel('Models', fontweight='bold')
        ax.set_ylabel('Test R² Score', fontweight='bold')
        ax.set_title('(A) Test R² Score Comparison', fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Strategy', frameon=True, fancybox=True, shadow=True, 
                 facecolor='white', edgecolor='gray', framealpha=0.9)
        ax.grid(True, alpha=0.3)
        ax.set_facecolor('#FAFAFA')
        
        # Test RMSE Plot
        ax = axes[0, 1]
        for i, method in enumerate(methods):
            bars = ax.bar(x + i*width, test_data[method]['rmse'], width, 
                         label=method, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.5)
            for bar, value in zip(bars, test_data[method]['rmse']):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                       f'{value:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        ax.set_xlabel('Models', fontweight='bold')
        ax.set_ylabel('Test RMSE (m)', fontweight='bold')
        ax.set_title('(B) Test RMSE Comparison', fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Strategy', frameon=True, fancybox=True, shadow=True, 
                 facecolor='white', edgecolor='gray', framealpha=0.9)
        ax.grid(True, alpha=0.3)
        ax.set_facecolor('#FAFAFA')
        
        # Test MAE Plot
        ax = axes[1, 0]
        for i, method in enumerate(methods):
            bars = ax.bar(x + i*width, test_data[method]['mae'], width, 
                         label=method, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.5)
            for bar, value in zip(bars, test_data[method]['mae']):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                       f'{value:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        ax.set_xlabel('Models', fontweight='bold')
        ax.set_ylabel('Test MAE (m)', fontweight='bold')
        ax.set_title('(C) Test MAE Comparison', fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Strategy', frameon=True, fancybox=True, shadow=True, 
                 facecolor='white', edgecolor='gray', framealpha=0.9)
        ax.grid(True, alpha=0.3)
        ax.set_facecolor('#FAFAFA')
        
        # Test MAPE Plot
        ax = axes[1, 1]
        for i, method in enumerate(methods):
            bars = ax.bar(x + i*width, test_data[method]['mape'], width, 
                         label=method, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.5)
            for bar, value in zip(bars, test_data[method]['mape']):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                       f'{value:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        ax.set_xlabel('Models', fontweight='bold')
        ax.set_ylabel('Test MAPE (%)', fontweight='bold')
        ax.set_title('(D) Test MAPE Comparison', fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(models)
        ax.legend(title='Strategy', frameon=True, fancybox=True, shadow=True, 
                 facecolor='white', edgecolor='gray', framealpha=0.9)
        ax.grid(True, alpha=0.3)
        ax.set_facecolor('#FAFAFA')
        
        # Style all axes
        for ax in axes.flat:
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('gray')
            ax.spines['bottom'].set_color('gray')
        
        plt.tight_layout(rect=[0, 0, 1, 0.93])
        plt.savefig(self.main_output_dir / 'comprehensive_metrics_comparison.png', 
                   dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
        plt.close(fig)
    
    def _print_final_summary(self):
        """Print comprehensive final summary."""
        print("\n" + "="*100)
        print("🎉 ANALYSIS COMPLETE - HYPERPARAMETER OPTIMIZATION COMPARISON")
        print("="*100)
        
        # Find overall best model by test RMSE
        best_result = min(self.all_results, key=lambda x: x['test_metrics']['rmse'])
        
        print(f"\n🏆 BEST OVERALL MODEL:")
        print(f"   Method: {best_result['method']}")
        print(f"   Model: {best_result['model_name']}")
        print(f"   Test R²: {best_result['test_metrics']['r2']:.3f}")
        print(f"   Test RMSE: {best_result['test_metrics']['rmse']:.2f} m")
        print(f"   Test MAE: {best_result['test_metrics']['mae']:.2f} m")
        print(f"   Test MAPE: {best_result['test_metrics']['mape']:.1f}%")
        print(f"   Training Time: {best_result['training_time']:.1f} seconds")
        if best_result['cv_rmse']:
            print(f"   CV RMSE: {best_result['cv_rmse']:.3f} m")
        
        # Summary by method
        print("\n📊 SUMMARY BY METHOD:")
        print("-" * 80)
        
        for method in ['Default', 'Grid Search + CV', 'Grid Search No CV']:
            method_results = [r for r in self.all_results if r['method'] == method]
            avg_r2 = np.mean([r['test_metrics']['r2'] for r in method_results])
            avg_rmse = np.mean([r['test_metrics']['rmse'] for r in method_results])
            avg_mae = np.mean([r['test_metrics']['mae'] for r in method_results])
            avg_mape = np.mean([r['test_metrics']['mape'] for r in method_results])
            avg_time = np.mean([r['training_time'] for r in method_results])
            
            print(f"\n{method}:")
            print(f"  Average Test R²: {avg_r2:.3f}")
            print(f"  Average Test RMSE: {avg_rmse:.2f} m")
            print(f"  Average Test MAE: {avg_mae:.2f} m")
            print(f"  Average Test MAPE: {avg_mape:.1f}%")
            print(f"  Average Training Time: {avg_time:.1f} seconds")
        
        # Model ranking
        print("\n🥇 MODEL RANKING BY TEST RMSE:")
        print("-" * 50)
        sorted_results = sorted(self.all_results, key=lambda x: x['test_metrics']['rmse'])
        for i, result in enumerate(sorted_results, 1):
            print(f"{i:2d}. {result['model_name']} ({result['method']}) - "
                  f"RMSE: {result['test_metrics']['rmse']:.3f} m, "
                  f"R²: {result['test_metrics']['r2']:.3f}")
        
        # Performance improvement analysis
        print("\n📈 PERFORMANCE IMPROVEMENT ANALYSIS:")
        print("-" * 60)
        
        default_results = {r['model_name']: r for r in self.all_results if r['method'] == 'Default'}
        
        for method in ['Grid Search + CV', 'Grid Search No CV']:
            method_results = {r['model_name']: r for r in self.all_results if r['method'] == method}
            print(f"\n{method} vs Default:")
            
            for model_name in default_results.keys():
                if model_name in method_results:
                    default_rmse = default_results[model_name]['test_metrics']['rmse']
                    method_rmse = method_results[model_name]['test_metrics']['rmse']
                    improvement = ((default_rmse - method_rmse) / default_rmse) * 100
                    
                    default_r2 = default_results[model_name]['test_metrics']['r2']
                    method_r2 = method_results[model_name]['test_metrics']['r2']
                    r2_improvement = ((method_r2 - default_r2) / default_r2) * 100
                    
                    print(f"  {model_name}:")
                    print(f"    RMSE improvement: {improvement:+.1f}%")
                    print(f"    R² improvement: {r2_improvement:+.1f}%")
        
        print("\n" + "="*100)

def main():
    """Main function to run the analysis."""
    # Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    # Create instance and run analysis
    analyzer = GroundwaterHyperparameterAnalysis(file_path)
    
    if analyzer.load_and_preprocess_data():
        analyzer.run_comprehensive_comparison(n_jobs=1)
    else:
        print("❌ Failed to load and preprocess data. Please check the file path and format.")

if __name__ == "__main__":
    main()